# 2VA — Simulação C: Ensemble (30 seeds)

**Cenário H1:** Transfer Learning (XLM-RoBERTa) + Transformers + Ensemble Learning  
Uma seed → N modelos treinados com sub-seeds → predições agregadas → métricas registradas  
Repete 30 vezes com as mesmas seeds do notebook 01 → permite comparação pareada (Wilcoxon)

## Passo 1 — Setup

In [ ]:
import sys
import random
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, random_split
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, mean_absolute_error

project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root))
sys.path.insert(1, str(project_root / "2VA" / "src"))

from src.models.classifier import HotelReviewClassifier
from src.models.trainer    import MultiTaskLoss, train_epoch
from src.data.dataset      import ReviewDataset
from ensemble              import HotelReviewEnsemble

# ── configuração ───────────────────────────────────────────────────────────────
SEEDS        = list(range(30))   # mesmas seeds do notebook 01 — pareamento garantido
N_MODELS     = 3                 # modelos por ensemble
N_EPOCHS     = 3
BATCH_SIZE   = 16
LR           = 2e-5
VAL_SPLIT    = 0.2

DATA_PATH    = project_root / "data" / "processed" / "reviews_labeled.csv"
TMP_DIR      = Path("../results/tmp")          # checkpoints temporários por simulação
RESULTS_PATH = Path("../results/ensemble_results.csv")
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

TMP_DIR.mkdir(parents=True, exist_ok=True)

print(f"device    : {DEVICE}")
print(f"seeds     : {SEEDS[0]}..{SEEDS[-1]}  ({len(SEEDS)} simulacoes)")
print(f"modelos   : {N_MODELS} por ensemble  |  epochs : {N_EPOCHS}")

## Passo 2 — Dados

Mesmo `make_loaders` do notebook 01. A seed principal controla o split — garante que ensemble e modelo único são avaliados **no mesmo conjunto de validação** para cada seed.

In [ ]:
full_dataset = ReviewDataset(str(DATA_PATH))

n_total = len(full_dataset)
n_val   = int(n_total * VAL_SPLIT)
n_train = n_total - n_val

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_loaders(seed: int):
    generator = torch.Generator().manual_seed(seed)
    train_ds, val_ds = random_split(full_dataset, [n_train, n_val], generator=generator)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, val_loader

print(f"total: {n_total}  |  treino: {n_train}  |  validacao: {n_val}")

## Passo 3 — `eval_ensemble` e `run_ensemble`

`eval_ensemble` avalia o ensemble em lote — usa `_aggregate` por batch em vez de `predict` (que processa um texto por vez e retorna strings).  
`run_ensemble` treina N modelos com sub-seeds, salva em disco, carrega o ensemble, avalia e limpa os checkpoints temporários.

In [ ]:
def eval_ensemble(ensemble: HotelReviewEnsemble, val_loader: DataLoader) -> dict:
    """Avalia o ensemble em lote — agrega tensores, não strings."""
    all_sentiment_preds, all_sentiment_labels = [], []
    all_rating_preds,    all_rating_labels    = [], []

    for batch in val_loader:
        input_ids      = batch["input_ids"].to(ensemble.device)
        attention_mask = batch["attention_mask"].to(ensemble.device)

        outputs_list = [
            ensemble._forward_one(m, input_ids, attention_mask)
            for m in ensemble.models
        ]
        aggregated = ensemble._aggregate(outputs_list)

        sentiment_preds = aggregated["sentiment"].argmax(dim=1)
        all_sentiment_preds.extend(sentiment_preds.cpu().tolist())
        all_sentiment_labels.extend(batch["label_sentiment"].tolist())

        rating_preds = aggregated["rating"].squeeze(1)
        all_rating_preds.extend(rating_preds.cpu().tolist())
        all_rating_labels.extend(batch["label_rating"].tolist())

    return {
        "f1_macro": f1_score(all_sentiment_labels, all_sentiment_preds, average="macro"),
        "mae":      mean_absolute_error(all_rating_labels, all_rating_preds),
    }


def run_ensemble(seed: int) -> dict:
    _, val_loader = make_loaders(seed)   # mesmo split que o notebook 01
    checkpoint_paths = []

    # treina N_MODELS modelos com sub-seeds derivadas da seed principal
    for i in range(N_MODELS):
        sub_seed = seed * N_MODELS + i   # ex: seed=2, i=1 → sub_seed=7 (único, sem colisão)
        set_seed(sub_seed)

        train_loader, _ = make_loaders(sub_seed)
        model     = HotelReviewClassifier().to(DEVICE)
        criterion = MultiTaskLoss()
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
        total_steps = N_EPOCHS * len(train_loader)
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps,
        )
        for _ in range(N_EPOCHS):
            train_epoch(model, train_loader, optimizer, criterion, DEVICE, scheduler)

        # salva checkpoint temporário
        ckpt_path = TMP_DIR / f"seed{seed}_model{i}.pt"
        torch.save(model.state_dict(), ckpt_path)
        checkpoint_paths.append(str(ckpt_path))

    # cria ensemble a partir dos checkpoints salvos e avalia
    ensemble = HotelReviewEnsemble(checkpoint_paths, device=DEVICE)
    metrics  = eval_ensemble(ensemble, val_loader)

    # limpa checkpoints temporários
    for path in checkpoint_paths:
        Path(path).unlink()

    return {
        "seed":     seed,
        "f1_macro": round(metrics["f1_macro"], 4),
        "mae":      round(metrics["mae"], 4),
    }

## Passo 4 — Loop das 30 simulações

In [ ]:
results = []

for i, seed in enumerate(SEEDS):
    print(f"[{i+1:02d}/30] seed={seed} ({N_MODELS} modelos) ...", end=" ", flush=True)
    row = run_ensemble(seed)
    results.append(row)
    print(f"F1={row['f1_macro']:.4f}  MAE={row['mae']:.4f}")

print("\nSimulacoes concluidas.")

## Passo 5 — Salvar CSV + preview

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df.to_csv(RESULTS_PATH, index=False)
print(f"Salvo em: {RESULTS_PATH}")

summary = df[["f1_macro", "mae"]].agg(["mean", "std", "min", "max"]).round(4)
print("\n--- Resumo das 30 simulacoes (C: ensemble de 3 modelos) ---")
print(summary.to_string())

df